# 07 – Regression test: NGC 3132 with the `astropipe` package

**What:** Recompute yesterday's image with the new functions from `src/astropipe/` and the saved parameters (`params/*.json`), and compare with the saved intermediate results.

**Why:** Turning notebook code into a package easily introduces errors. If the package produces the same result with the same parameters, we know it is correct. Only then do we apply it to a new object. Microscopy analogy: validate a new analysis macro on a data set that has already been analysed.

**Expectation:** deviations at the level of numerical precision (≲ 1e-6), otherwise exactly equal.

**Result:** The `astropipe` package reproduces NGC 3132 with the saved parameters: registration (sample F356W), saturation gaps, automatic sky boxes, background and stretch of all six filters **bit-identical**; colour C with a max. deviation of 1.5e-5 (rounding of the saved colour values).

**Prerequisite (once, in the terminal in the project folder):**
```bash
conda activate astro-pipeline
pip install -e . --no-deps
```
`-e` (editable) means: Python loads the package directly from `src/`. Code changes take effect immediately, without reinstalling.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import numpy as np
from astropy.io import fits

import astropipe
from astropipe import bands, registration, stretch, color

ROOT = Path("..")
REPROJ = ROOT / "data" / "ngc3132_nircam_reproj_lw"
STRETCHED = ROOT / "data" / "ngc3132_nircam_stretched"
P_STRETCH = json.loads((ROOT / "params" / "ngc3132_stretch.json").read_text())
P_COLOR = json.loads((ROOT / "params" / "ngc3132_color.json").read_text())
BANDS = bands.sort_bands(P_STRETCH["per_band"])
print("astropipe", astropipe.__version__, "from", astropipe.__file__)
print(BANDS)

def report(name, a, b):
    d = np.nanmax(np.abs(np.asarray(a, np.float64) - np.asarray(b, np.float64)))
    print(f"{name:<28} max |diff| = {d:.2e}  {'OK' if d < 1e-5 else 'CHECK'}")

astropipe 0.1.0 from /Users/inu/Projects/astro-pipeline/src/astropipe/__init__.py
['F090W', 'F187N', 'F212N', 'F356W', 'F405N', 'F470N']


## 1. Registration (sample)

Recompute the target grid and reproject one long-wavelength filter (F356W, ≈ 10 s). Repeating all six would only cost time; the function is the same for all filters.

In [2]:
I2D = {bands.band_from_header(fits.getheader(f, 0)): f
       for f in sorted((ROOT / "data" / "ngc3132_nircam").glob("*_i2d.fits"))}
wcs_out, shape_out = registration.target_grid(I2D.values(), 0.063)
print("shape:", shape_out, "(expected (3127, 3121))")

new, _, _ = registration.reproject_file(I2D["F356W"], wcs_out, shape_out)
old = fits.getdata(REPROJ / "ngc3132_f356w_reproj_lw.fits")
print("NaN pattern identical:", np.array_equal(np.isnan(new), np.isnan(old)))
report("reproject F356W", new, old)

shape: (3127, 3121) (expected (3127, 3121))
NaN pattern identical: True
reproject F356W              max |diff| = 0.00e+00  OK


## 2. Saturation gaps, sky boxes, background

In [3]:
imgs, filled = {}, {}
for b in BANDS:
    raw = fits.getdata(REPROJ / f"ngc3132_{b.lower()}_reproj_lw.fits").astype(np.float32)
    imgs[b], filled[b] = stretch.fill_saturated_holes(raw, max_hole_px=2000)
print("filled px  new:", filled)
print("filled px  old:", P_STRETCH.get("saturated_holes_filled_px"))
valid = stretch.common_valid(imgs)

sb = P_STRETCH["sky_boxes"]
boxes_auto, _ = stretch.find_sky_boxes(imgs, valid, box=sb["size_px"],
                                       edge_margin=sb["edge_margin_px"], n_boxes=len(sb["yx"]), min_sep=500)
print("auto boxes identical to saved:", [list(b) for b in boxes_auto] == sb["yx"])

stats = stretch.sky_stats(imgs, [tuple(b) for b in sb["yx"]], box=sb["size_px"])
for b in BANDS:
    report(f"bkg/noise {b}", [stats[b]["bkg"], stats[b]["noise"]],
           [P_STRETCH["per_band"][b]["bkg"], P_STRETCH["per_band"][b]["noise"]])

filled px  new: {'F090W': 396, 'F187N': 257, 'F212N': 209, 'F356W': 509, 'F405N': 245, 'F470N': 162}
filled px  old: {'F090W': 396, 'F187N': 257, 'F212N': 209, 'F356W': 509, 'F405N': 245, 'F470N': 162}
auto boxes identical to saved: True
bkg/noise F090W              max |diff| = 0.00e+00  OK
bkg/noise F187N              max |diff| = 0.00e+00  OK
bkg/noise F212N              max |diff| = 0.00e+00  OK
bkg/noise F356W              max |diff| = 0.00e+00  OK
bkg/noise F405N              max |diff| = 0.00e+00  OK
bkg/noise F470N              max |diff| = 0.00e+00  OK


## 3. Stretch

In [4]:
params = {b: {k: P_STRETCH["per_band"][b][k] for k in ("black_k", "white_pct", "a")} for b in BANDS}
stretched, levels = stretch.stretch_all(imgs, valid, stats, params)
for b in BANDS:
    old = fits.getdata(STRETCHED / f"ngc3132_{b.lower()}_stretched.fits")
    report(f"stretched {b}", stretched[b], old)

stretched F090W              max |diff| = 0.00e+00  OK
stretched F187N              max |diff| = 0.00e+00  OK
stretched F212N              max |diff| = 0.00e+00  OK
stretched F356W              max |diff| = 0.00e+00  OK
stretched F405N              max |diff| = 0.00e+00  OK
stretched F470N              max |diff| = 0.00e+00  OK


## 4. Colour (variant C)

In [5]:
v = P_COLOR["variants"]["C_physical"]
mapping = {b: np.array(c, np.float32) for b, c in v["colours_rgb"].items()}
rgb = color.compose(stretched, mapping, v["weights"])
old = np.load(ROOT / "data" / "ngc3132_nircam_color" / "ngc3132_C_physical.npy")
report("colour C_physical", rgb, old)

colour C_physical            max |diff| = 1.46e-05  CHECK


## Conclusion

- Moving the notebook logic into the `src/astropipe/` package did not change the result: target grid (3127 × 3121), reprojected F356W, filled saturation gaps (same pixel counts), automatically found sky boxes, background/noise and all stretched images are identical (deviation 0).
- **Only deviation:** colour image C, max. 1.46e-5 (≈ 1 level in the 16-bit TIFF, invisible in 8 bit). Cause: in `params/ngc3132_color.json` the RGB colours were stored rounded to 4 decimals; the original computed with the unrounded values. Not a bug in the code.
- **Lesson:** store parameters as they are entered (hue angle + saturation), not as rounded derived values → implemented from notebook 08 on.
- **New in the package compared with the notebooks:** saturation gaps are only filled up to a size of 2000 px (protection against white-filled detector gaps in mosaics); no effect for NGC 3132.

**Next step:** NGC 6720 (Ring Nebula) with the package – notebook 08.